# 第6章 环境绩效测度方法

> 由书稿国家样本代码适配生成（分省份）：读取 china data.xlsx 并将 dmuname 设为 '省份'。

In [ ]:
from deabook import data_path
from deabook import DEAweak
from deabook.constant import RTS_CRS, RTS_VRS1, RTS_VRS2, OPT_LOCAL


In [ ]:
# CRS
model = DEAweak.ushephard(data, sent="K L E=Y:CO2", rts=RTS_CRS)
res = model.optimize(solver="mosek")

# VRS1
model = DEAweak.ushephard(data, sent="K L E=Y:CO2", rts=RTS_VRS1)
res = model.optimize(solver="mosek")

# VRS2
model = DEAweak.ushephard(data, sent="K L E=Y:CO2", rts=RTS_VRS2)
res = model.optimize(solver="mosek")


In [ ]:
# CRS
model = DEAweak.hyper(data, sent="K L E=Y:CO2", rts=RTS_CRS)
res = model.optimize(solver="mosek")

# VRS1
model = DEAweak.hyper(data, sent="K L E=Y:CO2", rts=RTS_VRS1)
res = model.optimize(solver="mosek")

# VRS2
model = DEAweak.hyper(data, sent="K L E=Y:CO2", rts=RTS_VRS2)
res = model.optimize(solver="mosek")


In [ ]:
# CRS
model = DEAweak.ddf(data, sent="K L E=Y:CO2", rts=RTS_CRS)
res = model.optimize(solver="mosek")

# VRS1
model = DEAweak.ddf(data, sent="K L E=Y:CO2", rts=RTS_VRS1)
res = model.optimize(solver="mosek")

# VRS2
model = DEAweak.ddf(data, sent="K L E=Y:CO2", rts=RTS_VRS2)
res = model.optimize(solver="mosek")

# 显式给出方向向量（与默认一致：投入不变、扩张期望产出、压缩非期望产出）
model = DEAweak.ddf(data, sent="K L E=Y:CO2",
                    gx=[0,0,0], gy=[1], gb=[1], rts=RTS_CRS)
res = model.optimize(solver="mosek")

# 也可设定其他非负方向，例如同时收缩第一个投入 K（gx 取非零）：
#   model = DEAweak.ddf(data, sent="K L E=Y:CO2",
#                       gx=[1,0,0], gy=[1], gb=[1], rts=RTS_CRS)
# 方向改变后，EB/EO 的换算口径需按对应方向重新确定（本书案例沿用上述默认方向）


In [ ]:
# CRS
model = DEAweak.nddf(data, sent="K L E=Y:CO2", rts=RTS_CRS)
res = model.optimize(solver="mosek")

# VRS1
model = DEAweak.nddf(data, sent="K L E=Y:CO2", rts=RTS_VRS1)
res = model.optimize(solver="mosek")

# VRS2
model = DEAweak.nddf(data, sent="K L E=Y:CO2", rts=RTS_VRS2)
res = model.optimize(solver="mosek")

# 显式给出方向向量（与默认一致：投入不变、扩张期望产出、压缩非期望产出）
model = DEAweak.nddf(data, sent="K L E=Y:CO2",
                     gx=[0,0,0], gy=[1], gb=[1], rts=RTS_CRS)
res = model.optimize(solver="mosek")

# 也可设定其他非负方向，例如同时收缩第一个投入 K（gx 取非零）：
#   model = DEAweak.nddf(data, sent="K L E=Y:CO2",
#                        gx=[1,0,0], gy=[1], gb=[1], rts=RTS_CRS)
# 方向改变后，EB/EO/EBO 的换算口径需按对应方向重新确定（本书案例沿用上述默认方向）


In [ ]:
# 导入包含弱可处置性假设的DEA分析模块
from deabook import DEAweak
from deabook.constant import RTS_VRS1, RTS_VRS2, RTS_CRS, OPT_LOCAL

import pandas as pd

# 直接读取 OECD 国家数据（2015年后），含 33 个国家
dmuname = '省份'  # 决策单元标识字段名
data = (
    pd.read_excel(data_path("china data.xlsx"))
    .query('year>2015')
    .reset_index(drop=True)
)

# 构建时空标识数据
yearid = data[['year', dmuname]]  # 包含年份和国家名称
print("参与分析的决策单元：", yearid[dmuname].unique())  # 打印所有决策单元名称


In [ ]:
# 环境效率分析模型定义与执行

# 1. CRS 假设下的纯环境效率（ushephard，非期望产出导向）
def PEE_CRS():
    """规模报酬不变下的纯环境效率评估"""
    model = DEAweak.ushephard(data, sent="K L E=Y:CO2", rts=RTS_CRS)
    return model.optimize(solver="mosek")

# 2. VRS1 假设下的纯环境效率（同质弱处置）
def PEE_VRS1():
    """规模报酬可变-同质弱处置环境效率"""
    model = DEAweak.ushephard(data, sent="K L E=Y:CO2", rts=RTS_VRS1)
    return model.optimize(solver="mosek")

# 3. VRS2 假设下的纯环境效率（异质弱处置）
def PEE_VRS2():
    """规模报酬可变-异质弱处置环境效率"""
    model = DEAweak.ushephard(data, sent="K L E=Y:CO2", rts=RTS_VRS2)
    return model.optimize(solver="mosek")

# 执行三个模型
PEE_CRS = PEE_CRS()
PEE_VRS1 = PEE_VRS1()
PEE_VRS2 = PEE_VRS2()

# 结果处理与重命名：ushephard 仅返回 rho（PEE = rho），据方向换算
models = {
    'CRS': PEE_CRS.rename(columns={'rho':'PEE_CRS'}),
    'VRS1': PEE_VRS1.rename(columns={'rho':'PEE_VRS_SAME'}),
    'VRS2': PEE_VRS2.rename(columns={'rho':'PEE_VRS_DIFF'})
}
for key in models:
    models[key] = models[key].drop(columns=['optimization_status','objective_value'])

# 合并时空信息并生成透视表（列名映射与表号映射）
result_tables = {}
col_map = {'CRS': 'PEE_CRS', 'VRS1': 'PEE_VRS_SAME', 'VRS2': 'PEE_VRS_DIFF'}
table_num_map = {'CRS': 5, 'VRS1': 6, 'VRS2': 7}
for name, df in zip(['CRS','VRS1','VRS2'],
                   [pd.concat([models['CRS'],yearid],axis=1),
                   pd.concat([models['VRS1'],yearid],axis=1),
                   pd.concat([models['VRS2'],yearid],axis=1)]):
    result_tables[name] = df.pivot_table(
        values=[col_map[name]],
        index=dmuname,
        columns='year'
    ).to_markdown(f"table6.{table_num_map[name]}_DEA_UO_{name}.md")


In [ ]:
from deabook import DEAweak
from deabook.constant import RTS_VRS1, RTS_VRS2, RTS_CRS, OPT_LOCAL
import pandas as pd

# 1. CRS 混合环境效率模型（双曲 hyper，期望产出—非期望产出方向）
def HYPER_CRS():
    model = DEAweak.hyper(data,
                          sent="K L E=Y:CO2",
                          rts=RTS_CRS,
                          evalindex=None,
                          refindex=None)
    return model.optimize(solver="mosek")

# 2. VRS1 混合环境效率模型（双曲 hyper，同质弱处置）
def HYPER_VRS1():
    model = DEAweak.hyper(data,
                          sent="K L E=Y:CO2",
                          rts=RTS_VRS1,
                          evalindex=None,
                          refindex=None)
    return model.optimize(solver="mosek")

# 3. VRS2 混合环境效率模型（双曲 hyper，异质弱处置）
def HYPER_VRS2():
    model = DEAweak.hyper(data,
                          sent="K L E=Y:CO2",
                          rts=RTS_VRS2,
                          evalindex=None,
                          refindex=None)
    return model.optimize(solver="mosek")

HYPER_CRS_result = HYPER_CRS()
HYPER_VRS1_result = HYPER_VRS1()
HYPER_VRS2_result = HYPER_VRS2()

# 数据清洗：hyper 仅返回 rho，混合环境效率由调用方据方向换算
#   CRS: MEE_CRS = sqrt(rho)
#   VRS1/VRS2: MEE_BO = 1 - rho，MEE_O = 1/(1 + rho)
HYPER_CRS_clean = HYPER_CRS_result.drop(
    columns=['optimization_status', 'objective_value']
).assign(MEE_CRS=lambda d: d['rho'] ** 0.5).drop(columns=['rho'])

HYPER_VRS1_clean = HYPER_VRS1_result.drop(
    columns=['optimization_status', 'objective_value']
).assign(
    MEE_BO_VRS_SAME=lambda d: 1 - d['rho'],
    MEE_O_VRS_SAME=lambda d: 1 / (1 + d['rho'])
).drop(columns=['rho'])

HYPER_VRS2_clean = HYPER_VRS2_result.drop(
    columns=['optimization_status', 'objective_value']
).assign(
    MEE_BO_VRS_DIFF=lambda d: 1 - d['rho'],
    MEE_O_VRS_DIFF=lambda d: 1 / (1 + d['rho'])
).drop(columns=['rho'])

# 合并年份信息
HYPER_CRS_merged = pd.concat([HYPER_CRS_clean, yearid], axis=1)
HYPER_VRS1_merged = pd.concat([HYPER_VRS1_clean, yearid], axis=1)
HYPER_VRS2_merged = pd.concat([HYPER_VRS2_clean, yearid], axis=1)

# 生成透视表
HYPER_CRS_pivot = HYPER_CRS_merged.pivot_table(
    values=['MEE_CRS'],
    index=dmuname,
    columns='year'
)

HYPER_VRS1_pivot = HYPER_VRS1_merged.pivot_table(
    values=['MEE_BO_VRS_SAME', 'MEE_O_VRS_SAME'],
    index=dmuname,
    columns='year'
)

HYPER_VRS2_pivot = HYPER_VRS2_merged.pivot_table(
    values=['MEE_BO_VRS_DIFF', 'MEE_O_VRS_DIFF'],
    index=dmuname,
    columns='year'
)

# 输出结果
HYPER_CRS_pivot.to_markdown("table6.8_HYPER_CRS.md")
HYPER_VRS1_pivot.to_markdown("table6.9_HYPER_VRS_SAME.md")
HYPER_VRS2_pivot.to_markdown("table6.10_HYPER_VRS_DIFF.md")


In [ ]:
# 环境方向距离函数分析

# 1. CRS 环境方向距离函数模型（ddf，期望产出—非期望产出方向）
def DDF_CRS():
    model = DEAweak.ddf(data,
                        sent="K L E=Y:CO2",
                        rts=RTS_CRS,
                        evalindex=None,
                        refindex=None)
    return model.optimize(solver="mosek")

# 2. 同质弱处置 VRS 模型
def DDF_VRS_SAME():
    model = DEAweak.ddf(data,
                        sent="K L E=Y:CO2",
                        rts=RTS_VRS1,
                        evalindex=None,
                        refindex=None)
    return model.optimize(solver="mosek")

# 3. 异质弱处置 VRS 模型
def DDF_VRS_DIFF():
    model = DEAweak.ddf(data,
                        sent="K L E=Y:CO2",
                        rts=RTS_VRS2,
                        evalindex=None,
                        refindex=None)
    return model.optimize(solver="mosek")

# 执行模型计算
DDF_CRS_df = DDF_CRS()
DDF_VRS_SAME_df = DDF_VRS_SAME()
DDF_VRS_DIFF_df = DDF_VRS_DIFF()

# 由方向距离函数值 rho 换算环境效率 EB 与产出效率 EO
# （与 DEA.ddf 一致，DEAweak.ddf 仅返回 rho，效率需调用方按方向换算：
#  EB = 1 - rho 对应坏产出端，EO = 1/(1 + rho) 对应好产出端）
DDF_CRS_df['EB_CRS'] = 1 - DDF_CRS_df['rho']
DDF_CRS_df['EO_CRS'] = 1 / (1 + DDF_CRS_df['rho'])
DDF_VRS_SAME_df['EB_VRS_SAME'] = 1 - DDF_VRS_SAME_df['rho']
DDF_VRS_SAME_df['EO_VRS_SAME'] = 1 / (1 + DDF_VRS_SAME_df['rho'])
DDF_VRS_DIFF_df['EB_VRS_DIFF'] = 1 - DDF_VRS_DIFF_df['rho']
DDF_VRS_DIFF_df['EO_VRS_DIFF'] = 1 / (1 + DDF_VRS_DIFF_df['rho'])

# 清理辅助列（rho 已用于换算，可一并删除）
for df in [DDF_CRS_df, DDF_VRS_SAME_df, DDF_VRS_DIFF_df]:
    df.drop(columns=['optimization_status','rho','objective_value'],
        inplace=True)

# 合并时空信息
DDF_CRS_merged = pd.concat([DDF_CRS_df, yearid], axis=1)
DDF_VRS_SAME_merged = pd.concat([DDF_VRS_SAME_df, yearid], axis=1)
DDF_VRS_DIFF_merged = pd.concat([DDF_VRS_DIFF_df, yearid], axis=1)

# 创建透视表函数
def create_ddf_pivot(df, eff_cols, output_file):
    pivot = df.pivot_table(
        values=eff_cols,
        index=dmuname,
        columns='year',
        aggfunc='mean',
        fill_value=None,
        margins=False,
        dropna=True,
        margins_name='All',
        observed=False
    )
    pivot.to_markdown(output_file)
    return pivot

# 生成并保存所有结果表
DDF_CRS_table = create_ddf_pivot(
    DDF_CRS_merged, ['EB_CRS','EO_CRS'], "table6.11_DDF_CRS.md"
)
DDF_VRS_SAME_table = create_ddf_pivot(
    DDF_VRS_SAME_merged, ['EB_VRS_SAME',
        'EO_VRS_SAME'], "table6.12_DDF_VRS_SAME.md"
)
DDF_VRS_DIFF_table = create_ddf_pivot(
    DDF_VRS_DIFF_merged, ['EB_VRS_DIFF',
        'EO_VRS_DIFF'], "table6.13_DDF_VRS_DIFF.md"
)


In [ ]:

# 非径向方向距离函数(NDDF)环境效率分析

# 导入必要库
from deabook import DEAweak
from deabook.constant import RTS_VRS1, RTS_VRS2, RTS_CRS, OPT_LOCAL
import pandas as pd

# 1. CRS 假设下的非径向方向距离函数（nddf，期望产出—非期望产出方向）
def NDDF_CRS():
    """规模报酬不变下的非径向环境效率评估"""
    model = DEAweak.nddf(data, sent="K L E=Y:CO2", rts=RTS_CRS)
    return model.optimize(solver="mosek")

# 2. VRS1 假设下的非径向方向距离函数（同质弱处置）
def NDDF_VRS_SAME():
    """同质弱处置的非径向环境效率评估"""
    model = DEAweak.nddf(data, sent="K L E=Y:CO2", rts=RTS_VRS1)
    return model.optimize(solver="mosek")

# 3. VRS2 假设下的非径向方向距离函数（异质弱处置）
def NDDF_VRS_DIFF():
    """异质弱处置的非径向环境效率评估"""
    model = DEAweak.nddf(data, sent="K L E=Y:CO2", rts=RTS_VRS2)
    return model.optimize(solver="mosek")

# 执行模型计算
NDDF_CRS_df = NDDF_CRS()
NDDF_VRS_SAME_df = NDDF_VRS_SAME()
NDDF_VRS_DIFF_df = NDDF_VRS_DIFF()

# 由变量级缩放因子 beta 换算环境效率 EB、产出效率 EO 与相对排放强度 EBO
# （nddf 仅返回 beta_Y、beta_CO2 与 objective_value；标准等权 wy = wb = 0.5：
#  EB = 0.5·(1 - beta_CO2)、EO = 0.5/(1 + beta_Y)、EBO = EB/EO）
def _nddf_eff(df):
    df = df.copy()
    df['EB']  = 0.5 * (1 - df['beta_CO2'])
    df['EO']  = 0.5 / (1 + df['beta_Y'])
    df['EBO'] = df['EB'] / df['EO']
    return df

NDDF_CRS_df      = _nddf_eff(NDDF_CRS_df)
NDDF_VRS_SAME_df = _nddf_eff(NDDF_VRS_SAME_df)
NDDF_VRS_DIFF_df = _nddf_eff(NDDF_VRS_DIFF_df)

# 重命名并保留效率列
NDDF_CRS_df      = NDDF_CRS_df[['EB','EO','EBO']].rename(
    columns={'EB':'EB_CRS','EO':'EO_CRS','EBO':'EBO_CRS'})
NDDF_VRS_SAME_df = NDDF_VRS_SAME_df[['EB','EO','EBO']].rename(
    columns={'EB':'EB_VRS_SAME','EO':'EO_VRS_SAME','EBO':'EBO_VRS_SAME'})
NDDF_VRS_DIFF_df = NDDF_VRS_DIFF_df[['EB','EO','EBO']].rename(
    columns={'EB':'EB_VRS_DIFF','EO':'EO_VRS_DIFF','EBO':'EBO_VRS_DIFF'})

# 合并时空信息
NDDF_CRS_merged = pd.concat([NDDF_CRS_df, yearid], axis=1)
NDDF_VRS_SAME_merged = pd.concat([NDDF_VRS_SAME_df, yearid], axis=1)
NDDF_VRS_DIFF_merged = pd.concat([NDDF_VRS_DIFF_df, yearid], axis=1)

# 创建透视表并保存
def create_nddf_pivot(df, eff_cols, output_file):
    df.pivot_table(
        values=eff_cols,
        index=dmuname,
        columns='year',
        aggfunc='mean',
        dropna=True
    ).to_markdown(output_file)

create_nddf_pivot(
    NDDF_CRS_merged, ['EB_CRS','EO_CRS','EBO_CRS'], "table6.14_NDDF_CRS.md"
)
create_nddf_pivot(
    NDDF_VRS_SAME_merged,
    ['EB_VRS_SAME', 'EO_VRS_SAME', 'EBO_VRS_SAME'],
    "table6.15_NDDF_VRS_SAME.md"
)
create_nddf_pivot(
    NDDF_VRS_DIFF_merged,
    ['EB_VRS_DIFF', 'EO_VRS_DIFF', 'EBO_VRS_DIFF'],
    "table6.16_NDDF_VRS_DIFF.md"
)


In [ ]:
# 计算每个模型在不同年份的平均环境效率（基于已生成的透视表，按列求均值得年度均值）
hyper_mean_year = HYPER_VRS1_pivot['MEE_BO_VRS_SAME'].mean()  # HYPER_VRS_SAME 模型的年度均值
nddf_mean_year = NDDF_VRS_SAME_merged.groupby('year')['EB_VRS_SAME'].mean()
# NDDF_VRS_SAME 模型的年度均值
ddf_mean_year = DDF_VRS_SAME_merged.groupby('year')['EB_VRS_SAME'].mean()
# DDF_VRS_SAME 模型的年度均值

xtick2 = tuple(nddf_mean_year.index)  # 生成 x 轴刻度标签
nddf_mean_year  # 输出 NDDF_VRS_SAME 模型的平均值
ddf_mean_year   # 输出 DDF_VRS_SAME 模型的平均值

# 导入所需的绘图库
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib notebook

# 设置图表字体为 SimHei，以支持中文显示
from matplotlib import rcParams
rcParams['font.family'] = 'SimHei'

# 导入 numpy 用于数值操作
import numpy as np

# 创建图形和坐标轴对象
fig, ax = plt.subplots()  # 创建图形网格

# 隐藏图表上边框和右边框
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

################################ 绘图部分 #################################

n_groups = 4  # 4 个年份（2016—2019）
index = np.arange(n_groups)  # x 轴位置索引

# 绘制 HYPER_VRS_SAME（深蓝虚线，方块标记）
rects1 = ax.plot(index, hyper_mean_year,
                 color='darkblue', alpha=0.8, linestyle="dashed", marker="s",
                 label='HYPER_VRS_SAME'
                 )

# 绘制 NDDF_VRS_SAME（橄榄绿虚线，三角标记）
rects2 = ax.plot(index, nddf_mean_year,
                 color='olivedrab', alpha=0.8, linestyle="dashed", marker="^",
                 label='NDDF_VRS_SAME'
                 )

########################################### 结束 ################################

# 设置 y 轴标签
ax.set_ylabel(r'$\mathregular{环境效率}$')

# 设置 x 轴的刻度标签
ax.set_xticks(index)
ax.set_xticklabels(xtick2)

# 显示图例
plt.legend()

# 设置 x 轴刻度的旋转角度为 80 度，避免重叠
plt.xticks(rotation=80)

# 保存图表（OECD 国家样本）
plt.savefig("分年份环境效率-国.png", dpi=900, bbox_inches='tight')  # 保存为高分辨率图像

# 显示图形
plt.show()


In [ ]:
# 从 HYPER_VRS1_pivot 透视表中提取 'MEE_BO_VRS_SAME' 列并计算各国家（行）的平均值
hyper_mean_cou = HYPER_VRS1_pivot['MEE_BO_VRS_SAME'].mean(axis=1)
# 从 NDDF_VRS_SAME_merged 数据中按国家计算 'EB_VRS_SAME' 列的平均值
nddf_mean_cou = NDDF_VRS_SAME_merged.groupby(dmuname)['EB_VRS_SAME'].mean()
# 将两个均值序列合并成一个 DataFrame，并按第一个列（hyper_mean_cou）排序
zz = pd.concat([hyper_mean_cou, nddf_mean_cou], axis=1).sort_values(0)

# 重新提取排序后的数据
hyper_mean_cou = zz[0]
nddf_mean_cou = zz[1]
# 获取排序后的索引
xtick2 = tuple(nddf_mean_cou.index)
# 创建图形和坐标轴
fig, ax = plt.subplots()  

# 隐藏顶部和右侧边框
ax.spines['top'].set_visible(False)  
ax.spines['right'].set_visible(False)

################################ 绘图 #################################
# OECD 国家样本共 33 个决策单元
n_groups = 33

# 生成 x 轴的位置索引（0 到 n_groups-1）
index = np.arange(n_groups)
bar_width = 0.5  # 柱子宽度

# 绘制 HYPER_VRS_SAME（深蓝柱）
rects1 = ax.bar(
    index - bar_width/4,  # 两柱并列居中，HYPER 略左
    hyper_mean_cou,
    bar_width/2,
    color='darkblue',
    alpha=0.8,
    edgecolor='white',
    label='HYPER_VRS_SAME'
)

# 绘制 NDDF_VRS_SAME（橄榄绿柱）
rects2 = ax.bar(
    index + bar_width/4,  # NDDF 略右
    nddf_mean_cou,
    bar_width/2,
    color='olivedrab',
    alpha=0.8,
    edgecolor='white',
    label='NDDF_VRS_SAME'
)

###########################################

# 设置 y 轴标签（显示为“环境效率”）
ax.set_ylabel(r'$\mathregular{环境效率}$')  
# 设置 x 轴刻度位置
ax.set_xticks(index)  
# 设置 x 轴刻度标签（省份或国家名称）
ax.set_xticklabels(xtick2)  
# 添加图例（显示在左上角）
plt.legend(loc='upper left')  
# 旋转 x 轴标签 80 度，避免重叠
plt.xticks(rotation=80)  

# 保存图片（OECD 国家样本）
plt.savefig("分国家环境效率2.png", dpi=900, bbox_inches='tight')

# 显示图形
plt.show()
